# Aula 03: Teoria Geral dos Dispositivos Industriais (Sensores Avançados, Atuadores e Controladores CLP/IHM)

## 1. Visão Geral e Escopo Técnico
Esta aula consolida a **fundamentação teórica dos dispositivos físicos do chão de fábrica**: sensores fotoelétricos, ultrassônicos, identificação por RFID, atuadores pneumáticos/elétricos e a arquitetura determinística do **Controlador Lógico Programável (CLP)** com seu **Ciclo de SCAN**.

### Objetivos de Aprendizagem:
- Compreender a operação dos sensores fotoelétricos (Barreira, Retroreflexivo com Filtro Polarizado e Difuso com Supressão de Fundo).
- Analisar o funcionamento dos sensores ultrassônicos por tempo de voo (Time-of-Flight) e entender o conceito de Zona Cega.
- Compreender a arquitetura de identificação RFID para rastreabilidade e genealogia industrial.
- Compreender o funcionamento de atuadores pneumáticos (válvulas solenoide 5/2 vias) e acionamentos elétricos (Inversores VFD e Servomotores).
- Dominar as 4 etapas do **Ciclo de SCAN do CLP** e executar simulações computacionais em Python.

## 2. Sensores Avançados e Sistemas de Identificação

![Comparação de Sensores Fotoelétricos](img/sensores_fotoeletricos.jpg)

### 2.1 Sensores Fotoelétricos (Ópticos)
Utilizam feixes de luz visível ou infravermelha modulada para detectar objetos sem contato físico:

1. **Sistema por Barreira (Through-Beam):** Emissor e receptor em invólucros separados instalados de frente um para o outro. Maior alcance (até 50 metros) e imune à cor ou brilho da peça.
2. **Sistema Retroreflexivo (Retro-reflective):** Emissor e receptor no mesmo corpo apontando para um espelho prismático. Utiliza **filtros de polarização cruzada** em 90 graus para evitar falsas leituras por superfícies metálicas brilhantes.
3. **Sistema Difuso (Diffuse-Reflective):** O feixe reflete na própria superfície da peça. Sensores difusos avançados possuem **Supressão de Fundo (PSD)** para detectar objetos escuros próximos ignorando um fundo claro distante.

### 2.2 Sensores Ultrassônicos (Acústicos)
Emitem pulsos de ondas de som de alta frequência (200 kHz a 400 kHz) e medem o tempo de retorno do eco (**Time-of-Flight - ToF**):

- **Vantagens:** Capazes de detectar materiais transparentes (garrafas PET, vidro, água limpa) inacessíveis aos sensores ópticos.
- **Compensação Térmica:** Possuem um termistor NTC para corrigir a velocidade do som no ar ($v \approx 331,5 + 0,6 \cdot T$).
- **Zona Cega (Blind Zone):** Região física nos primeiros centímetros à frente da face sensora (5 cm a 20 cm) onde a recepção do eco não pode ocorrer devido ao amortecimento da vibração piezoelétrica.

### 2.3 Identificação Automática por Rádio Frequência (RFID)
Permite a leitura e escrita sem contato de dados armazenados em tags acopladas a peças, ferramentas ou pallets na esteira Smart N1:

- **Tag / Transponder:** Etiqueta contendo um código UID (*Unique Identifier*) inalterável e memória de usuário regravável.
- **Leitor / Escritor RFID:** Antena industrial conectada ao CLP via rede (Profinet, Modbus TCP) para registrar o histórico de produção (genealogia do produto).

## 3. Atuadores Industriais e Controladores (CLP)

### 3.1 Atuadores Pneumáticos e Elétricos
- **Cilindros Pneumáticos de Dupla Ação:** Controlados por **válvulas solenoide 5/2 vias** (5 vias de pressão/exaustão e 2 posições).
- **Inversores de Frequência (VFD):** Modulam a velocidade de motores trifásicos alterando a frequência da tensão ($V/f = \text{constante}$).
- **Servomotores:** Motores síncronos de alta precisão com **Encoder** de realimentação de posição para malhas fechadas de controle.

### 3.2 O Ciclo de SCAN do Controlador Lógico Programável (CLP)

![Ciclo de SCAN do CLP](img/ciclo_de_scan_clp.jpg)

O CLP opera de forma determinística e cíclica repetindo continuamente 4 etapas fundamentais:

1. **Leitura das Entradas (Read Inputs):** Copia o estado elétrico físico dos sensores das cartas de entrada para a Memória de Imagem do Processo de Entrada (PII).
2. **Execução do Programa do Usuário (Execute Logic):** Avalia os degraus de lógica (Ladder, Texto Estruturado) linha por linha, atualizando as variáveis internas.
3. **Atualização das Saídas (Write Outputs):** Escreve os valores armazenados na Memória de Imagem de Saída (PIQ) para as cartas físicas de atuadores.
4. **Housekeeping e Comunicação de Rede:** Executa diagnósticos de hardware e atende a requisições de comunicação de rede (Modbus, Profinet, IHM).

## 4. Passo a Passo Prático para o Estudante

Siga os passos a seguir para simular o comportamento do Ciclo de SCAN de um CLP e a gravação de dados em Tags RFID industriais.

### Passo 1: Executar o Simulador do Ciclo de SCAN do CLP em Python
Rode a célula abaixo para observar o determinismo do ciclo de leitura, lógica e atuação.

In [ ]:
# Simulador do Ciclo de SCAN de um CLP Industrial
import time

class CLP_Simulador:
    def __init__(self):
        self.pii = {"S1_SENSO_INDUTIVO": False, "EMERGENCIA_NF": True}
        self.piq = {"SOLENOIDE_ESTEIRA": False, "ALARME_SONORO": False}
        self.scan_time_ms = 0.0
        
    def ciclo_scan(self, entrada_física_s1, entrada_fisica_emergencia):
        t_inicio = time.perf_counter()
        
        # 1. Leitura das Entradas Fisicas para PII
        self.pii["S1_SENSO_INDUTIVO"] = entrada_física_s1
        self.pii["EMERGENCIA_NF"] = entrada_fisica_emergencia
        
        # 2. Execucao da Logica de Controle
        if not self.pii["EMERGENCIA_NF"]:
            self.piq["SOLENOIDE_ESTEIRA"] = False
            self.piq["ALARME_SONORO"] = True
        elif self.pii["S1_SENSO_INDUTIVO"]:
            self.piq["SOLENOIDE_ESTEIRA"] = True
            self.piq["ALARME_SONORO"] = False
        else:
            self.piq["SOLENOIDE_ESTEIRA"] = False
            self.piq["ALARME_SONORO"] = False
            
        # 3. Escrita das Saidas (PIQ -> Fisico)
        t_fim = time.perf_counter()
        self.scan_time_ms = (t_fim - t_inicio) * 1000
        return self.piq, self.scan_time_ms

clp = CLP_Simulador()
print("=== SIMULAÇÃO DE CICLO DE SCAN DO CLP ===\n")
res_saidas, scan_t = clp.ciclo_scan(entrada_física_s1=True, entrada_fisica_emergencia=True)
print(f"Estado das Saidas no PIQ: {res_saidas}")
print(f"Tempo de SCAN (SCAN Time): {scan_t:.4f} ms")

### Passo 2: Simular a Leitura e Gravação de Tag RFID em Python
Rode a célula a seguir para simular a escrita do histórico de produção na memória de uma Tag RFID durante a passagem por uma estação.

In [ ]:
# Simulador de Leitura e Escrita de Tag RFID Industrial

class TagRFID:
    def __init__(self, uid):
        self.uid = uid
        self.user_memory = {}
        
    def escrever_memoria(self, lote, serial, status_qc):
        self.user_memory["LOTE"] = lote
        self.user_memory["SERIAL"] = serial
        self.user_memory["STATUS_QC"] = status_qc
        self.user_memory["TIMESTAMP_GRAVACAO"] = time.strftime("%Y-%m-%d %H:%M:%S")

tag_pallet = TagRFID(uid="E004015029A84F2B")
tag_pallet.escrever_memoria(lote="LOTE-2026-N1", serial="SN-99842", status_qc="APROVADO")

print("=== LEITURA DE MEMÓRIA DA TAG RFID NA ESTAÇÃO DE EMBALAGEM ===")
print(f"UID Inalteravel: {tag_pallet.uid}")
print("Dados Registrados na Memoria do Pallet:")
for k, v in tag_pallet.user_memory.items():
    print(f"  - {k}: {v}")

## 5. Exercícios de Fixação e Avaliação

1. Explique o funcionamento do filtro de polarização em sensores fotoelétricos retroreflexivos e por que ele previne erros de contagem em latas de alumínio ou objetos brilhantes.
2. Um sensor ultrassônico é instalado acima de um tanque de óleo para medição de nível. Por que o nível do líquido não pode se aproximar a menos de 10 cm da face do sensor?
3. Quais são as 4 etapas do Ciclo de SCAN de um CLP e por que a leitura de entradas e atualização de saídas ocorrem em instantes consolidados em vez de assincronamente durante a execução da lógica?